<img align="left" src = https://project.lsst.org/sites/default/files/Rubin-O-Logo_0.png width=250 style="padding: 10px"> 
<br>
<b>PPDB interface example</b> <br>
Contact author: Ian Sullivan<br>
Last verified to run: 11 June 2026<br>
LSST Science Piplines version: w_2026_24<br>
Run at USDF on LSSTCam repo embargo

# 1. Main package imports

In [1]:
import os
import importlib
import pprint

import matplotlib.pyplot as plt
#%matplotlib widget
%matplotlib inline

import math
import numpy as np
import pandas as pd
import scipy
import astropy.units as u
from astropy.coordinates import SkyCoord

In [2]:
import lsst.afw.image as afwImage
import lsst.afw.display as afwDisplay
import lsst.geom
import lsst.afw.geom as afwGeom

import lsst.daf.butler as dafButler
import lsst.pipe.base

In [3]:
from lsst.analysis.ap import apdb
from lsst.analysis.ap import nb_utils
from astropy.table import Table

In [4]:
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

In [5]:
afwDisplay.setDefaultBackend("matplotlib")

# Get a butler

In [6]:
!eups list afw

   gf03f0b42f3+e27ba6bf39 	current d_2026_05_30 d_2026_06_01 d_2026_06_02 d_2026_06_03 d_latest w_2026_23 w_latest setup


In [7]:
repo = 'embargo'

butler_registry = dafButler.Butler(repo)

In [8]:
skymap = "lsst_cells_v2"

In [9]:
skyMap = butler_registry.get("skyMap", collections="skymaps", skymap=skymap)

In [10]:
def find_datasets(butler, dataset, collectionPattern="*"):
    collections = []
    for info in butler_registry.collections.query_info(collectionPattern, include_summary=True):
        if dataset in info.dataset_types:
            collections.append(info.name)
    collections.sort()
    for c in collections:
        print(c)

In [11]:
find_datasets(butler_registry, "preliminary_visit_image", collectionPattern="*")

LSSTCam/prompt/output-2025-12-30/daytime
LSSTCam/prompt/output-2025-12-30/daytime/20260108T171841Z
LSSTCam/prompt/output-2025-12-30/daytime/20260109T005233Z
LSSTCam/prompt/output-2025-12-30/daytime/20260109T013703Z
LSSTCam/prompt/output-2025-12-31/daytime
LSSTCam/prompt/output-2025-12-31/daytime/20260110T013026Z
LSSTCam/prompt/output-2025-12-31/daytime/20260113T225743Z
LSSTCam/prompt/output-2025-12-31/daytime/20260113T233447Z
LSSTCam/prompt/output-2025-12-31/daytime/20260114T091952Z
LSSTCam/prompt/output-2025-12-31/daytime/20260114T183230Z
LSSTCam/prompt/output-2025-12-31/daytime/20260114T194016Z
LSSTCam/prompt/output-2025-12-31/daytime/20260114T210842Z
LSSTCam/prompt/output-2026-01-01/daytime
LSSTCam/prompt/output-2026-01-01/daytime/20260114T222501Z
LSSTCam/prompt/output-2026-01-05/daytime
LSSTCam/prompt/output-2026-01-05/daytime/20260115T000909Z
LSSTCam/prompt/output-2026-01-06/daytime
LSSTCam/prompt/output-2026-01-06/daytime/20260115T014148Z
LSSTCam/prompt/output-2026-01-07/daytime


In [12]:
collection = "LSSTCam/runs/prompt/20260607/ApPipe/pipelines-9e8db3d-config-8f017ea"

In [13]:
butler = dafButler.Butler(repo, collections=collection, instrument='LSSTCam')

In [14]:
datarefs = butler.query_datasets("preliminary_visit_image", limit=-100000)

In [15]:
datarefs[630]

DatasetRef(DatasetType('preliminary_visit_image', {band, instrument, day_obs, detector, physical_filter, visit}, ExposureF), {instrument: 'LSSTCam', detector: 18, visit: 2026060700418, band: 'i', day_obs: 20260607, physical_filter: 'i_39'}, run='LSSTCam/runs/prompt/20260607/ApPipe/pipelines-9e8db3d-config-8f017ea', id=019ea559-ec0d-78f2-b79e-fbdb6bc6ecf6)

In [16]:
len(datarefs)

1726

In [17]:
visits = []
for ref in datarefs:
    dataId = ref.dataId
    visits.append(dataId['visit'])
visits = set(visits)
visits

{2026060700180,
 2026060700183,
 2026060700414,
 2026060700415,
 2026060700416,
 2026060700417,
 2026060700418,
 2026060700419,
 2026060700420,
 2026060700421,
 2026060700422,
 2026060700423,
 2026060700424,
 2026060700427,
 2026060700428,
 2026060700429,
 2026060700430,
 2026060700431,
 2026060700432,
 2026060700450,
 2026060700474,
 2026060700475}

In [18]:
solar_refs = butler.query_datasets("ss_source_detector")

In [19]:
len(solar_refs)

1634

In [20]:
visit = 2026060700431
detector = 80

## Find a detector and visit for a given diaSourceId

Note that the unpacker needs any dataId with the right instrument, it does *not* need to match the detector or visit (or dataset type!) of the diaSource

In [21]:
from lsst.ip.diffim.detectAndMeasure import DetectAndMeasureConfig
from lsst.meas.base import IdGenerator
dataId2 = butler.registry.expandDataId(dataId)
config = DetectAndMeasureConfig()
unpacker = IdGenerator.unpacker_from_config(config.idGenerator, fixed=dataId2)
print(unpacker(169878951476854863))

(1, {instrument: 'LSSTCam', detector: 179, visit: 2026011500051}, 79)


In [22]:
print(unpacker(170032911482355882))

(1, {instrument: 'LSSTCam', detector: 33, visit: 2026021900263}, 170)


# Set up a PPDB interface

To set up a data-int TAP access token:
1. The token must come from data-int (not USDF), with the read:tap scope.
    1. Log in to https://data-int.lsst.cloud with your Rubin credentials.
    2. Open the token page: click your username (top-right) → Security tokens, or go straight to https://data-int.lsst.cloud/auth/tokens.
    3. Click Create Token.
    4. Give it a name (e.g. ppdb-tap), check the read:tap scope, and set an expiration.
    5. Click Create and copy the token immediately — it's shown only once. It looks like gt-XX….
2.  Make it available to the tool
    PpdbTap reads RSP_TOKEN from the environment (or takes token=). Pick one:
    * Option A — paste per session (recommended; nothing written to disk):
        - `import os, getpass`
        - `os.environ["RSP_TOKEN"] = getpass.getpass("data-int RSP token: ")`
        - Paste at the prompt. It lives only in the kernel and disappears when it stops.
    * Option B — store in a private file (persists across sessions). In a USDF RSP terminal:
        - Create a file `~/.rsp_data_int_token` with permissions mode 600, paste the token as the (only) contents

In [23]:
# Set the token as an environment variable
os.environ["RSP_TOKEN"] = open(os.path.expanduser("~/.rsp_data_int_token")).read().strip()

In [24]:
# Run a quick check to verify the connection with the token
from lsst.analysis.ap.ppdb import PpdbTap
ppdb = PpdbTap()
ppdb.run_query("SELECT TOP 1 diaObjectId FROM ppdb.DiaObject")   

diaObjectId
int64
170503492472406050


In [26]:
# Unpack the diaObjectId
print(unpacker(170503492472406050))

(1, {instrument: 'LSSTCam', detector: 185, visit: 2026060600188}, 34)


## Load all diaObjects and diaSources overlapping an exposure

In [27]:
science = butler.get("preliminary_visit_image", visit=visit, detector=detector)

obj_on_image    = ppdb.load_objects(exposure=science, padding=1.0)

In [28]:
obj_on_image

diaObjectId,validityStartMjdTai,validityEndMjdTai,ra,raErr,dec,decErr,ra_dec_Cov,u_psfFluxMean,u_psfFluxMeanErr,u_psfFluxSigma,u_psfFluxNdata,u_fpFluxMean,u_fpFluxMeanErr,g_psfFluxMean,g_psfFluxMeanErr,g_psfFluxSigma,g_psfFluxNdata,g_fpFluxMean,g_fpFluxMeanErr,r_psfFluxMean,r_psfFluxMeanErr,r_psfFluxSigma,r_psfFluxNdata,r_fpFluxMean,r_fpFluxMeanErr,i_psfFluxMean,i_psfFluxMeanErr,i_psfFluxSigma,i_psfFluxNdata,i_fpFluxMean,i_fpFluxMeanErr,z_psfFluxMean,z_psfFluxMeanErr,z_psfFluxSigma,z_psfFluxNdata,z_fpFluxMean,z_fpFluxMeanErr,y_psfFluxMean,y_psfFluxMeanErr,y_psfFluxSigma,y_psfFluxNdata,y_fpFluxMean,y_fpFluxMeanErr,u_scienceFluxMean,u_scienceFluxMeanErr,g_scienceFluxMean,g_scienceFluxMeanErr,r_scienceFluxMean,r_scienceFluxMeanErr,i_scienceFluxMean,i_scienceFluxMeanErr,z_scienceFluxMean,z_scienceFluxMeanErr,y_scienceFluxMean,y_scienceFluxMeanErr,u_psfFluxMin,u_psfFluxMax,u_psfFluxMaxSlope,u_psfFluxErrMean,g_psfFluxMin,g_psfFluxMax,g_psfFluxMaxSlope,g_psfFluxErrMean,r_psfFluxMin,r_psfFluxMax,r_psfFluxMaxSlope,r_psfFluxErrMean,i_psfFluxMin,i_psfFluxMax,i_psfFluxMaxSlope,i_psfFluxErrMean,z_psfFluxMin,z_psfFluxMax,z_psfFluxMaxSlope,z_psfFluxErrMean,y_psfFluxMin,y_psfFluxMax,y_psfFluxMaxSlope,y_psfFluxErrMean,firstDiaSourceMjdTai,lastDiaSourceMjdTai,nDiaSources
,,,deg,deg,deg,deg,deg2,nJy,nJy,nJy,,nJy,nJy,nJy,nJy,nJy,,nJy,nJy,nJy,nJy,nJy,,nJy,nJy,nJy,nJy,nJy,,nJy,nJy,nJy,nJy,nJy,,nJy,nJy,nJy,nJy,nJy,,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy / d,nJy,nJy,nJy,nJy / d,nJy,nJy,nJy,nJy / d,nJy,nJy,nJy,nJy / d,nJy,nJy,nJy,nJy / d,nJy,nJy,nJy,nJy / d,nJy,,,
int64,float64,float64,float64,float32,float64,float32,float32,float32,float32,float32,int32,float32,float32,float32,float32,float32,int32,float32,float32,float32,float32,float32,int32,float32,float32,float32,float32,float32,int32,float32,float32,float32,float32,float32,int32,float32,float32,float32,float32,float32,int32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float64,float64,int32
170437563324039561,61199.16950586233,--,260.17398792649317,4.9237283e-06,-13.493796755228663,5.6051167e-06,-1.6054368e-12,--,--,--,0,--,--,--,--,--,0,--,--,--,--,--,0,--,--,-70205.91,710.6695,6752.867,2,--,--,110972.5,2358.1597,--,1,--,--,--,--,--,0,--,--,--,--,--,--,--,--,738283.3,624.48206,996082.1,2249.9294,--,--,--,--,--,--,--,--,--,--,--,--,--,--,-74971.78,-65421.785,639.12524,1005.0398,110972.5,110972.5,--,2358.1597,--,--,--,--,--,--,3
170437563325087766,61186.1955613483,--,260.21646274935006,2.1923004e-05,-13.578951917422716,2.2537106e-05,-6.1743002e-12,--,--,--,0,--,--,--,--,--,0,--,--,--,--,--,0,--,--,--,--,--,0,--,--,-23800.385,593.58704,35160.766,2,--,--,--,--,--,0,--,--,--,--,--,--,--,--,--,--,173817.34,559.515,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,-29942.916,19781.916,-17050.182,1161.4568,--,--,--,--,--,--,2
170437563325612051,61186.1955613483,--,260.1934314886691,3.149999e-07,-13.45188168076886,3.3003684e-07,-2.1656392e-15,--,--,--,0,--,--,--,--,--,0,--,--,--,--,--,0,--,--,--,--,--,0,--,--,-95356.47,1531.5916,54816.793,2,--,--,--,--,--,0,--,--,--,--,--,--,--,--,--,--,1.6554742e+06,1372.199,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,-117523.586,-40000.934,-26581.799,2338.345,--,--,--,--,--,--,2
170437563325612091,61183.278808263414,--,260.19878607058695,1.6447478e-06,-13.33879726306283,1.809865e-06,-1.2666594e-13,--,--,--,0,--,--,--,--,--,0,--,--,--,--,--,0,--,--,--,--,--,0,--,--,448455.22,3234.7227,--,1,--,--,--,--,--,0,--,--,--,--,--,--,--,--,--,--,2.5016978e+06,3298.3127,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,448455.22,448455.22,--,3234.7227,--,--,--,--,--,--,1
170437563325612225,61183.278808263414,--,260.33283047122177,2.478124e-07,-13.521738820447018,2.5448352e-07,-7.4

## DiaSources and DiaForcedSources must be loaded from their associated diaObjects
There are utilities for loading DiaSources and DiaForcedSources directly, but those won't be available through the public PPDB interface

In [29]:
src_on_image    = ppdb.load_sources(diaObjectId=obj_on_image['diaObjectId'])
forced_on_image = ppdb.load_forced_sources(diaObjectId=obj_on_image['diaObjectId'])

In [30]:
src_on_image

diaSourceId,visit,detector,diaObjectId,ssObjectId,parentDiaSourceId,ssObjectReassocTimeMjdTai,midpointMjdTai,ra,raErr,dec,decErr,ra_dec_Cov,x,xErr,y,yErr,centroid_flag,apFlux,apFluxErr,apFlux_flag,apFlux_flag_apertureTruncated,isNegative,snr,psfFlux,psfFluxErr,psfLnL,psfChi2,psfNdata,psfFlux_flag,psfFlux_flag_edge,psfFlux_flag_noGoodPixels,trailFlux,trailFluxErr,trailRa,trailRaErr,trailDec,trailDecErr,trailLength,trailLengthErr,trailAngle,trailAngleErr,trailChi2,trailNdata,trail_flag_edge,dipoleMeanFlux,dipoleMeanFluxErr,dipoleFluxDiff,dipoleFluxDiffErr,dipoleLength,dipoleAngle,dipoleChi2,dipoleNdata,scienceFlux,scienceFluxErr,forced_PsfFlux_flag,forced_PsfFlux_flag_edge,forced_PsfFlux_flag_noGoodPixels,templateFlux,templateFluxErr,ixx,iyy,ixy,ixxPSF,iyyPSF,ixyPSF,shape_flag,shape_flag_no_pixels,shape_flag_not_contained,shape_flag_parent_source,extendedness,reliability,band,isDipole,dipoleFitAttempted,timeProcessedMjdTai,timeWithdrawnMjdTai,bboxSize,pixelFlags,pixelFlags_bad,pixelFlags_cr,pixelFlags_crCenter,pixelFlags_edge,pixelFlags_nodata,pixelFlags_nodataCenter,pixelFlags_interpolated,pixelFlags_interpolatedCenter,pixelFlags_offimage,pixelFlags_saturated,pixelFlags_saturatedCenter,pixelFlags_suspect,pixelFlags_suspectCenter,pixelFlags_streak,pixelFlags_streakCenter,pixelFlags_injected,pixelFlags_injectedCenter,pixelFlags_injected_template,pixelFlags_injected_templateCenter,glint_trail
,,,,,,,d,deg,deg,deg,deg,deg2,pix,pix,pix,pix,,nJy,nJy,,,,,nJy,nJy,,,,,,,nJy,nJy,deg,deg,deg,deg,arcsec,arcsec,deg,deg,,,,nJy,nJy,nJy,nJy,arcsec,deg,,,nJy,nJy,,,,nJy,nJy,arcsec2,arcsec2,arcsec2,arcsec2,arcsec2,arcsec2,,,,,,,,,,,,pix,,,,,,,,,,,,,,,,,,,,,
int64,int64,int16,int64,int64,int64,float64,float64,float64,float32,float64,float32,float32,float32,float32,float32,float32,bool,float32,float32,bool,bool,bool,float32,float32,float32,float32,float32,int32,bool,bool,bool,float32,float32,float64,float32,float64,float32,float32,float32,float32,float32,float32,int32,bool,float32,float32,float32,float32,float32,float32,float32,int32,float32,float32,bool,bool,bool,float32,float32,float32,float32,float32,float32,float32,float32,bool,bool,bool,bool,float32,float32,str1,bool,bool,float64,float64,int32,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool
170437563325612091,2026052200498,77,170437563325612091,--,0,--,61183.2769292288,260.19878607058695,1.6447478e-06,-13.33879726306283,1.809865e-06,-1.2666594e-13,3816.717,0.032737266,726.47687,0.029459354,False,433544.22,3687.0095,False,False,False,139.18414,448455.22,3234.7227,--,1872.8823,1681,False,False,False,423340.2,2639.3418,260.19878680905225,--,-13.338797499350205,--,0.57020885,0.09236729,-24.39379,9.543466,--,0,False,--,--,--,--,--,--,--,0,2.5016978e+06,3298.3127,False,False,False,2.0379248e+06,130.82973,0.28892705,0.2637479,-0.005799346,0.31851104,0.28921688,0.001778932,False,False,False,False,0.060485758,1.0,z,False,False,61183.27874216692,--,48,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
170437563325612410,2026052200498,77,170437563325612410,--,0,--,61183.2769292288,260.1785039108946,5.8703617e-06,-13.356699471603905,5.3218355e-06,1.355057e-12,3574.7603,0.09538872,312.21826,0.10611001,False,-209859.81,3807.387,False,False,True,55.360283,-187507.97,3388.408,--,2008.1133,1681,False,False,False,-197000.61,3345.2292,260.1785009861189,--,-13.356700112328392,--,1.3987781,0.123884514,43.816246,5.661841,--,0,False,--,--,--,--,--,--,--,0,2.7767015e+06,3419.444,False,False,False,2.967878e+06,175.17691,0.38459635,0.4555672,0.08597317,0.3202063,0.29003748,0.0024817113,False,False,False,False,0.6616933,0.8344213,z,False,False,61183.27874216692,--,39,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
170437563325612405,2026052200498,77,170437563325612405,--,0,--,61183.2769292288,260.1672099023549,3.9

In [40]:
forced_on_image

diaForcedSourceId,diaObjectId,ra,dec,visit,detector,psfFlux,psfFluxErr,midpointMjdTai,scienceFlux,scienceFluxErr,band,timeProcessedMjdTai,timeWithdrawnMjdTai
,,deg,deg,,,nJy,nJy,d,nJy,nJy,,,
int64,int64,float64,float64,int64,int16,float32,float32,float64,float32,float32,str1,float64,float64
170441938410405889,170437563325612428,260.2427225775508,-13.36921659692646,2026052300327,57,-7851.028,405.6288,61184.22477168911,83020.27,346.15182,i,61184.226733420684,--
170441938411978753,170437563324039561,260.1739823884213,-13.49380425299299,2026052300327,60,-76304.31,1024.6218,61184.22477168911,736374.1,878.3135,i,61184.22724247111,--
170441938410930177,170437563325612479,260.3760352569839,-13.311438158318731,2026052300327,58,128579.55,972.02,61184.22477168911,638796.25,827.40106,i,61184.22639995608,--
170450753548713986,170441938412505387,260.4505144827507,-13.415185959324718,2026052500469,31,-11623.966,642.6379,61186.191865819885,120674.93,571.56415,z,61186.19415987049,--
170450753548713985,170437563325612479,260.3760352569839,-13.311438158318731,2026052500469,31,26247.06,1147.0098,61186.191865819885,645050.9,1035.3059,z,61186.19415987049,--
170450753549762562,170437563325087766,260.21646386141936,-13.578952570886694,2026052500469,33,-29877.332,722.3503,61186.191865819885,171353.55,640.26874,z,61186.19394592251,--
170450753549762564,170441938411980148,260.2597116239107,-13.559975450536296,2026052500469,33,5387.3228,586.205,61186.191865819885,72349.92,519.286,z,61186.19394592251,--
170450753993834497,170437563325612428,260.24272423145266,-13.36922112429586,2026052500472,112,-18215.53,540.71765,61186.19331044428,89731.79,491.71118,z,61186.19551142314,--


In [41]:
obj_on_image[obj_on_image["nDiaSources"] > 2]

diaObjectId,validityStartMjdTai,validityEndMjdTai,ra,raErr,dec,decErr,ra_dec_Cov,u_psfFluxMean,u_psfFluxMeanErr,u_psfFluxSigma,u_psfFluxNdata,u_fpFluxMean,u_fpFluxMeanErr,g_psfFluxMean,g_psfFluxMeanErr,g_psfFluxSigma,g_psfFluxNdata,g_fpFluxMean,g_fpFluxMeanErr,r_psfFluxMean,r_psfFluxMeanErr,r_psfFluxSigma,r_psfFluxNdata,r_fpFluxMean,r_fpFluxMeanErr,i_psfFluxMean,i_psfFluxMeanErr,i_psfFluxSigma,i_psfFluxNdata,i_fpFluxMean,i_fpFluxMeanErr,z_psfFluxMean,z_psfFluxMeanErr,z_psfFluxSigma,z_psfFluxNdata,z_fpFluxMean,z_fpFluxMeanErr,y_psfFluxMean,y_psfFluxMeanErr,y_psfFluxSigma,y_psfFluxNdata,y_fpFluxMean,y_fpFluxMeanErr,u_scienceFluxMean,u_scienceFluxMeanErr,g_scienceFluxMean,g_scienceFluxMeanErr,r_scienceFluxMean,r_scienceFluxMeanErr,i_scienceFluxMean,i_scienceFluxMeanErr,z_scienceFluxMean,z_scienceFluxMeanErr,y_scienceFluxMean,y_scienceFluxMeanErr,u_psfFluxMin,u_psfFluxMax,u_psfFluxMaxSlope,u_psfFluxErrMean,g_psfFluxMin,g_psfFluxMax,g_psfFluxMaxSlope,g_psfFluxErrMean,r_psfFluxMin,r_psfFluxMax,r_psfFluxMaxSlope,r_psfFluxErrMean,i_psfFluxMin,i_psfFluxMax,i_psfFluxMaxSlope,i_psfFluxErrMean,z_psfFluxMin,z_psfFluxMax,z_psfFluxMaxSlope,z_psfFluxErrMean,y_psfFluxMin,y_psfFluxMax,y_psfFluxMaxSlope,y_psfFluxErrMean,firstDiaSourceMjdTai,lastDiaSourceMjdTai,nDiaSources
,,,deg,deg,deg,deg,deg2,nJy,nJy,nJy,,nJy,nJy,nJy,nJy,nJy,,nJy,nJy,nJy,nJy,nJy,,nJy,nJy,nJy,nJy,nJy,,nJy,nJy,nJy,nJy,nJy,,nJy,nJy,nJy,nJy,nJy,,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy,nJy / d,nJy,nJy,nJy,nJy / d,nJy,nJy,nJy,nJy / d,nJy,nJy,nJy,nJy / d,nJy,nJy,nJy,nJy / d,nJy,nJy,nJy,nJy / d,nJy,,,
int64,float64,float64,float64,float32,float64,float32,float32,float32,float32,float32,int32,float32,float32,float32,float32,float32,int32,float32,float32,float32,float32,float32,int32,float32,float32,float32,float32,float32,int32,float32,float32,float32,float32,float32,int32,float32,float32,float32,float32,float32,int32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float64,float64,int32
170437563324039561,61199.16950586233,--,260.17398792649317,4.9237283e-06,-13.493796755228663,5.6051167e-06,-1.6054368e-12,--,--,--,0,--,--,--,--,--,0,--,--,--,--,--,0,--,--,-70205.91,710.6695,6752.867,2,--,--,110972.5,2358.1597,--,1,--,--,--,--,--,0,--,--,--,--,--,--,--,--,738283.3,624.48206,996082.1,2249.9294,--,--,--,--,--,--,--,--,--,--,--,--,--,--,-74971.78,-65421.785,639.12524,1005.0398,110972.5,110972.5,--,2358.1597,--,--,--,--,--,--,3
170437563325612428,61186.19551149523,--,260.24272423145266,1.7991999e-05,-13.36922112429586,2.2857223e-05,-4.4086575e-11,--,--,--,0,--,--,--,--,--,0,--,--,--,--,--,0,--,--,-7749.0786,387.36755,--,1,--,--,-18007.607,503.05353,277.62918,2,--,--,--,--,--,0,--,--,--,--,--,--,--,--,80390.99,340.64893,90616.6,469.44025,--,--,--,--,--,--,--,--,--,--,--,--,--,--,-7749.079,-7749.079,--,387.36755,-18356.252,-17963.625,134.62813,1018.4436,--,--,--,--,--,--,3
170437563325612479,61199.16872967467,--,260.3760340545989,1.6538412e-06,-13.311437501633865,1.7332698e-06,-5.9273536e-14,--,--,--,0,--,--,--,--,--,0,--,--,--,--,--,0,--,--,98711.82,654.24927,35555.47,2,--,--,-271029.9,1798.3584,--,1,--,--,--,--,--,0,--,--,--,--,--,--,--,--,611382.5,573.77094,363396.84,1821.6869,--,--,--,--,--,--,--,--,--,--,--,--,--,--,74442.46,124725.484,-3365.148,925.66614,-271029.9,-271029.9,--,1798.3584,--,--,--,--,--,--,3
170441938411979562,61199.169533134,--,260.2634315443959,1.2256264e-05,-13.481978887890376,1.4941493e-05,-1.1372155e-11,--,--,--,0,--,--,--,--,--,0,--,--,--,--,--,0,--,--,3285.794,229.93549,136.02898,2,--,--,3006.1602,462.9558,--,1,--,--,--,--,--,0,--,--,--,--,--,--,--,--,32718.273,198.89206,41409.746,418.40137,--,--,--,--,--,--,--,--,--,--,--,--,--,--,3182.152,3374.5261,-12.874466,325.91364,3006.1602,3006.16

# Plot sources on top of images (note: better plots are coming soon, these are just a quick diagnostic)

In [50]:
def get_xy_from_source_table(table, wcs, degrees=False):
    ra = table['ra']
    dec = table['dec']
    x, y = wcs.skyToPixelArray(ra, dec, degrees=degrees)
    return Table.from_pandas(pd.DataFrame({'x':x, 'y':y}))

In [51]:
diaObject_xy = get_xy_from_source_table(obj_on_image, science.wcs, degrees=True)

In [42]:
diffim = butler.get("difference_image", visit=visit, detector=detector)
standardized = butler.get("dia_source_detector", visit=visit, detector=detector)

In [54]:
disp = afwDisplay.Display(figsize=(12,12), backend='firefly')
disp.mtv(diffim)

In [55]:
for x,y in zip(standardized["x"],standardized["y"]):
    disp.dot("+", x, y, size=10, ctype="blue")

In [56]:
for x,y in zip(diaObject_xy["x"],diaObject_xy["y"]):
    disp.dot("o", x, y, size=15, ctype="red")

The catalog diaSources and PPDB DiaObjects do align in Firefly (you'll have to take my word for it)